# Step 12 — Broader tumor/epithelial rescue across melanoma, NSCLC, and CRC

This notebook broadens the current `Tumor` call while preserving all existing
T, NK, B/plasma, myeloid, fibroblast/stromal, and endothelial assignments.

The current preliminary annotation leaves a large `Other/unresolved`
compartment. The earlier tumor hierarchy used relatively narrow epithelial and
melanoma programs plus a strict winner-takes-all lineage rule. That is useful
for specificity, but it can miss:

```text
NSCLC:
    alveolar, club, ciliated, basal/squamous, and poorly differentiated
    epithelial states

CRC:
    enterocyte/colonocyte, goblet, REG4-like, mucinous, and invasive
    epithelial states

melanoma:
    differentiated melanocytic and MITF-low/AXL-like invasive states
```

## What this notebook changes

Fresh UCell scores are calculated from the Step 03A ResolVI-corrected matrix
for an expanded marker panel.

Only cells currently labeled:

```text
Other/unresolved
```

are eligible for rescue. Existing immune, stromal, endothelial, and T-cell
labels are never overwritten.

The primary output uses one deliberately broad category:

```text
Tumor/epithelial
```

because this transcript-only pass is not intended to separate malignant cells
from normal epithelium. A secondary program column records which signature
provided the strongest evidence.

## Nested rescue tiers

```text
existing_tumor
    the original Step 11 Tumor cells

primary_rescue
    expanded tumor/epithelial score is sample-high
    + raw marker or epithelial-reference support
    + score is reasonably competitive with non-tumor programs

exploratory_only
    milder score threshold
    + at least one raw marker or epithelial-reference support
    + broader competition allowance
```

The recommended deliverable column is:

```python
prelim_cell_type_primary_tumor_expanded
```

The maximum-sensitivity counterpart is:

```python
prelim_cell_type_exploratory_tumor_expanded
```

## Important melanoma safeguard

The invasive melanoma program contains genes that can also occur in fibroblasts
or myeloid cells. It is allowed to drive a tumor rescue only when at least one
of the following is also present:

```text
melanoma-lineage raw marker
melanoma-lineage corrected score
epithelial/melanoma reference label
```

## Outputs

For each sample:

```text
full annotated H5AD
cell-level metadata Parquet
spatial metadata Parquet
rescued-cell ID table
before/after count table
signature-gene coverage table
score and spatial diagnostic figures
```

Across samples:

```text
before/after tumor and unresolved counts
new sample composition tables
patient and sample stacked bar charts
primary versus exploratory rescue comparison
```

ResolVI is not retrained. Only a small number of new UCell signatures are
calculated from the existing corrected Zarr.

In [1]:
# ---------------------------------------------------------------------
# Environment selection — run before importing pyUCell/Torch
# ---------------------------------------------------------------------
import os

GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("OMP_NUM_THREADS", "16")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "16")
os.environ.setdefault("MKL_NUM_THREADS", "16")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "16")

print("CUDA_VISIBLE_DEVICES:", GPU_ID)

CUDA_VISIBLE_DEVICES: 0


In [2]:
# ---------------------------------------------------------------------
# Imports and pyUCell compatibility inspection
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import inspect
import json
import math
import re
import traceback
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import zarr

try:
    import pyucell as uc

    PYUCELL_AVAILABLE = True
    PYUCELL_VERSION = importlib.metadata.version(
        "pyucell"
    )
    UCELL_SIGNATURE = inspect.signature(
        uc.compute_ucell_scores
    )
    UCELL_PARAMETERS = UCELL_SIGNATURE.parameters
    UCELL_ACCEPTS_VAR_KWARGS = any(
        parameter.kind
        is inspect.Parameter.VAR_KEYWORD
        for parameter in UCELL_PARAMETERS.values()
    )
except Exception as exc:
    uc = None
    PYUCELL_AVAILABLE = False
    PYUCELL_VERSION = None
    UCELL_SIGNATURE = None
    UCELL_PARAMETERS = {}
    UCELL_ACCEPTS_VAR_KWARGS = False
    PYUCELL_IMPORT_ERROR = (
        f"{type(exc).__name__}: {exc}"
    )

print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyUCell available:", PYUCELL_AVAILABLE)

if PYUCELL_AVAILABLE:
    print("pyUCell:", PYUCELL_VERSION)
    print("pyUCell signature:", UCELL_SIGNATURE)
else:
    print("pyUCell import error:", PYUCELL_IMPORT_ERROR)

anndata: 0.12.11
numpy: 2.4.4
pandas: 2.3.3
pyUCell available: True
pyUCell: 0.5.0
pyUCell signature: (adata: anndata._core.anndata.AnnData, signatures: dict[str, list[str]], layer: str = None, max_rank: int = 1500, ties_method: str = 'average', missing_genes: str = 'impute', chunk_size: int = 500, w_neg: float = 1.0, suffix: str = '_UCell', n_jobs: int = -1)


/tmp/ipykernel_95195/2313266393.py:51: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

PRELIMINARY_ROOT = (
    PIPELINE_ROOT
    / "11_preliminary_cell_type_annotation"
)
ALLCELL_ZARR_ROOT = (
    PIPELINE_ROOT
    / "03a_resolvi_allcell_zarr"
)

OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "12_broader_tumor_epithelial_rescue"
)
FIGURE_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"

for path in (
    OUTPUT_ROOT,
    FIGURE_ROOT,
    TABLE_ROOT,
):
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen",
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15",
    },
}

SECTION_NAMES = list(
    SAMPLE_INFO
)
CANCER_TYPE_ORDER = [
    "melanoma",
    "NSCLC",
    "colon_cancer",
]
BIOPSY_STAGE_ORDER = [
    "Screen",
    "C2D15",
]

CORRECTED_LAYER = (
    "resolvi_corrected_10k"
)

# pyUCell execution.
PREFER_DEVICE = "cuda:0"
UCELL_MAX_RANK = 1_500
UCELL_OUTER_CELL_CHUNK = 5_000
UCELL_INNER_CHUNK_SIZE = 1_000
UCELL_CPU_N_JOBS = min(
    8,
    os.cpu_count() or 1,
)
UCELL_GPU_FALLBACK_TO_CPU = True

# Resume from score staging to avoid repeating pyUCell after interruption.
RESUME_FROM_SCORE_PARQUET = True
OVERWRITE_EXPANDED_TUMOR_SCORES = False

# Primary and exploratory rescue thresholds.
PRIMARY_TUMOR_QUANTILE = 0.85
EXPLORATORY_TUMOR_QUANTILE = 0.75

PRIMARY_SCORE_FLOOR = 0.03
EXPLORATORY_SCORE_FLOOR = 0.02

PRIMARY_COMPETITION_ALLOWANCE = 0.10
EXPLORATORY_COMPETITION_ALLOWANCE = 0.20

PRIMARY_MIN_RAW_GENES = 2
EXPLORATORY_MIN_RAW_GENES = 1

# A single high-specificity raw marker can satisfy the raw-support requirement.
ALLOW_ONE_HIGH_SPECIFICITY_RAW_MARKER = True

# Pan-Human/Azimuth epithelial or melanoma labels can provide orthogonal support.
USE_REFERENCE_SUPPORT = True

# Only current Other/unresolved cells can be newly rescued.
ELIGIBLE_CURRENT_LABEL = (
    "Other/unresolved"
)

# Label used after broadening. This avoids claiming malignant versus normal
# epithelial separation from transcript signatures alone.
EXPANDED_TUMOR_LABEL = (
    "Tumor/epithelial"
)

WRITE_ANNOTATED_H5AD = True
WRITE_METADATA_PARQUET = True
WRITE_SPATIAL_PARQUET = True
WRITE_RESCUED_CELL_IDS = True
VALIDATE_WRITTEN_H5AD = True

H5AD_COMPRESSION = "lzf"
PLOT_DPI = 500
PLOT_MAX_CELLS = 200_000
RANDOM_STATE = 0

CONTINUE_ON_ERROR = True

PIPELINE_VERSION = (
    "2026-08-03-broader-tumor-epithelial-rescue-v1"
)

print("Samples:", SECTION_NAMES)
print("Output root:", OUTPUT_ROOT)

Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue


In [4]:
# ---------------------------------------------------------------------
# Expanded marker signatures
# ---------------------------------------------------------------------
# Shared epithelial architecture. This intentionally includes differentiated
# and basal keratin programs and does not attempt malignant/normal separation.
PAN_EPITHELIAL_EXPANDED = [
    "EPCAM",
    "TACSTD2",
    "KRT5",
    "KRT6A",
    "KRT6B",
    "KRT7",
    "KRT8",
    "KRT14",
    "KRT15",
    "KRT17",
    "KRT18",
    "KRT19",
    "KRT20",
    "CLDN4",
    "CLDN7",
    "CDH1",
    "JUP",
    "DSP",
    "DSG2",
    "DSC2",
    "MUC1",
    "SFN",
    "ITGB6",
    "MMP7",
]

# Lung adenocarcinoma, squamous/basal, alveolar, club, and ciliated epithelial
# states. Normal epithelium is deliberately included in the same broad class.
NSCLC_LUNG_EPITHELIAL = [
    "EPCAM",
    "TACSTD2",
    "KRT5",
    "KRT6A",
    "KRT6B",
    "KRT7",
    "KRT8",
    "KRT14",
    "KRT15",
    "KRT17",
    "KRT18",
    "KRT19",
    "TP63",
    "SFN",
    "DSG3",
    "DSC3",
    "CLDN18",
    "SLC34A2",
    "NAPSA",
    "SFTPA1",
    "SFTPA2",
    "SFTPB",
    "SFTPC",
    "SCGB1A1",
    "FOXJ1",
    "PIFO",
    "CAPS",
    "TPPP3",
    "CEACAM5",
    "CEACAM6",
    "MUC1",
    "MSLN",
    "MMP7",
    "ITGB6",
    "GPRC5A",
]

# Broad colorectal epithelial states: absorptive/colonocyte, goblet/mucinous,
# REG4-like, and invasive/poorly differentiated epithelial programs.
CRC_COLORECTAL_EPITHELIAL = [
    "EPCAM",
    "TACSTD2",
    "KRT8",
    "KRT18",
    "KRT19",
    "KRT20",
    "CEACAM5",
    "CEACAM6",
    "MUC1",
    "MUC13",
    "TFF3",
    "PHGR1",
    "LGALS4",
    "SLC26A3",
    "FABP1",
    "CDX2",
    "SATB2",
    "PIGR",
    "MMP7",
    "REG4",
    "SPINK4",
    "FCGBP",
    "MUC2",
    "AGR2",
]

# Differentiated and lineage-associated melanoma markers.
MELANOMA_LINEAGE_EXPANDED = [
    "MLANA",
    "PMEL",
    "TYR",
    "DCT",
    "MITF",
    "SOX10",
    "S100B",
    "TYRP1",
    "GPR143",
    "RAB38",
    "SLC45A2",
    "MIA",
    "PRAME",
    "CSPG4",
    "BIRC7",
    "EDNRB",
    "TRPM1",
    "OCA2",
    "MC1R",
]

# MITF-low/AXL-like support. This module is never sufficient by itself.
MELANOMA_INVASIVE_SUPPORT = [
    "AXL",
    "NGFR",
    "SOX9",
    "WNT5A",
    "FOSL1",
    "JUN",
    "EGFR",
    "ITGA3",
]

SIGNATURES_BY_CANCER = {
    "NSCLC": {
        "Pan_epithelial_expanded": (
            PAN_EPITHELIAL_EXPANDED
        ),
        "NSCLC_lung_epithelial": (
            NSCLC_LUNG_EPITHELIAL
        ),
    },
    "colon_cancer": {
        "Pan_epithelial_expanded": (
            PAN_EPITHELIAL_EXPANDED
        ),
        "CRC_colorectal_epithelial": (
            CRC_COLORECTAL_EPITHELIAL
        ),
    },
    "melanoma": {
        "Pan_epithelial_expanded": (
            PAN_EPITHELIAL_EXPANDED
        ),
        "Melanoma_lineage_expanded": (
            MELANOMA_LINEAGE_EXPANDED
        ),
        "Melanoma_invasive_support": (
            MELANOMA_INVASIVE_SUPPORT
        ),
    },
}

HIGH_SPECIFICITY_MARKERS = {
    "NSCLC": [
        "EPCAM",
        "TACSTD2",
        "KRT19",
        "KRT17",
        "SLC34A2",
        "NAPSA",
        "CEACAM5",
        "MSLN",
        "MMP7",
        "ITGB6",
    ],
    "colon_cancer": [
        "EPCAM",
        "KRT20",
        "CEACAM5",
        "CEACAM6",
        "MUC13",
        "TFF3",
        "PHGR1",
        "SLC26A3",
        "SATB2",
        "CDX2",
        "REG4",
        "MMP7",
    ],
    "melanoma": [
        "MLANA",
        "PMEL",
        "TYR",
        "DCT",
        "SOX10",
        "MITF",
        "GPR143",
        "MIA",
        "PRAME",
        "TYRP1",
        "SLC45A2",
    ],
}

# Existing non-tumor scores are used only as a competition audit.
COMPETING_SCORE_COLUMNS = {
    "T_core": "fallback_T_core_score",
    "Endothelial": (
        "fallback_Endothelial_score"
    ),
    "Monocyte_macrophage": (
        "fallback_Monocyte_macrophage_score"
    ),
    "Fibroblast": (
        "fallback_Fibroblast_score"
    ),
    "B_plasma": (
        "fallback_B_plasma_score"
    ),
    "NK": "fallback_NK_score",
}

In [5]:
# ---------------------------------------------------------------------
# Paths, hashing, and portable H5AD writing
# ---------------------------------------------------------------------
def paths_for_sample(
    sample: str,
) -> dict[str, Path]:
    source = (
        PRELIMINARY_ROOT
        / sample
        / f"{sample}_preliminary_cell_type_annotated.h5ad"
    )
    zarr_path = (
        ALLCELL_ZARR_ROOT
        / sample
        / f"{sample}_resolvi_allcells.zarr"
    )
    zarr_summary = (
        ALLCELL_ZARR_ROOT
        / sample
        / f"{sample}_resolvi_allcells_zarr_summary.json"
    )

    out = OUTPUT_ROOT / sample
    figures = out / "figures"

    out.mkdir(
        parents=True,
        exist_ok=True,
    )
    figures.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "source": source,
        "zarr": zarr_path,
        "zarr_summary": zarr_summary,
        "out": out,
        "figures": figures,
        "score_staging": (
            out
            / f"{sample}_expanded_tumor_scores.parquet"
        ),
        "coverage": (
            out
            / f"{sample}_expanded_tumor_signature_coverage.csv"
        ),
        "thresholds": (
            out
            / f"{sample}_expanded_tumor_thresholds.csv"
        ),
        "rescued_ids": (
            out
            / f"{sample}_expanded_tumor_rescued_cell_ids.csv"
        ),
        "counts": (
            out
            / f"{sample}_expanded_tumor_before_after_counts.csv"
        ),
        "metadata": (
            out
            / f"{sample}_expanded_tumor_metadata.parquet"
        ),
        "spatial": (
            out
            / f"{sample}_expanded_tumor_spatial.parquet"
        ),
        "annotated": (
            out
            / f"{sample}_preliminary_tumor_expanded.h5ad"
        ),
        "summary": (
            out
            / f"{sample}_expanded_tumor_summary.json"
        ),
    }


def names_hash(
    values,
) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(
            str(value).encode(
                "utf-8"
            )
        )
        digest.update(
            b"\0"
        )
    return digest.hexdigest()


def write_json(
    payload,
    path: str | Path,
) -> None:
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    temporary.replace(
        path
    )


def _is_nullable_string_series(
    series: pd.Series,
) -> bool:
    return (
        isinstance(
            series.dtype,
            pd.StringDtype,
        )
        or type(
            series.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
        or str(
            series.dtype
        ) == "str"
        or str(
            series.dtype
        ).startswith(
            "string"
        )
    )


def _legacy_string_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    for column in output.columns:
        series = output[
            column
        ]

        if _is_nullable_string_series(
            series
        ):
            output[
                column
            ] = (
                series
                .fillna("")
                .astype(str)
                .astype(object)
            )
            continue

        if pd.api.types.is_object_dtype(
            series.dtype
        ):
            def _safe_text(value):
                if (
                    value is None
                    or value is pd.NA
                ):
                    return ""
                if isinstance(
                    value,
                    bytes,
                ):
                    return value.decode(
                        "utf-8",
                        errors="replace",
                    )
                if isinstance(
                    value,
                    str,
                ):
                    return value
                if isinstance(
                    value,
                    np.generic,
                ):
                    value = value.item()
                if isinstance(
                    value,
                    (
                        dict,
                        list,
                        tuple,
                        set,
                        np.ndarray,
                        Path,
                    ),
                ):
                    return json.dumps(
                        value,
                        default=str,
                        sort_keys=True,
                    )
                return str(
                    value
                )

            output[
                column
            ] = (
                series
                .map(
                    _safe_text
                )
                .astype(object)
            )

    if (
        str(
            output.index.dtype
        ) == "str"
        or str(
            output.index.dtype
        ).startswith(
            "string"
        )
        or type(
            output.index.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
    ):
        name = (
            output.index.name
        )
        output.index = pd.Index(
            pd.Series(
                output.index,
                dtype="string",
            )
            .fillna("")
            .astype(str)
            .to_numpy(
                dtype=object
            ),
            name=name,
        )

    return output


def safe_write_h5ad(
    adata_object: ad.AnnData,
    filename: str | Path,
    *,
    compression: str = "lzf",
) -> None:
    filename = Path(
        filename
    )
    filename.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    adata_object.obs = (
        _legacy_string_frame(
            adata_object.obs
        )
    )
    adata_object.var = (
        _legacy_string_frame(
            adata_object.var
        )
    )

    temporary = filename.with_name(
        f"{filename.stem}.tmp{filename.suffix}"
    )
    if temporary.exists():
        temporary.unlink()

    with ad.settings.override(
        allow_write_nullable_strings=False
    ):
        adata_object.write_h5ad(
            temporary,
            compression=compression,
            convert_strings_to_categoricals=False,
        )

    temporary.replace(
        filename
    )
    print("Saved:", filename)


def choose_spatial_key(
    adata: ad.AnnData,
) -> str | None:
    for key in (
        "X_spatial",
        "spatial",
        "spatial_fullres",
    ):
        if key not in adata.obsm:
            continue

        value = np.asarray(
            adata.obsm[
                key
            ]
        )
        if (
            value.ndim == 2
            and value.shape[0]
            == adata.n_obs
            and value.shape[1]
            >= 2
        ):
            return key

    return None

In [6]:
# ---------------------------------------------------------------------
# pyUCell compatibility
# ---------------------------------------------------------------------
def _ucell_accepts(
    name: str,
) -> bool:
    return (
        name in UCELL_PARAMETERS
        or UCELL_ACCEPTS_VAR_KWARGS
    )


def map_genes(
    var_names: pd.Index,
    genes: list[str],
) -> list[str]:
    lookup = {}
    for value in (
        var_names.astype(str)
    ):
        lookup.setdefault(
            value.upper(),
            value,
        )

    output = []
    for gene in genes:
        mapped = lookup.get(
            str(gene).upper()
        )
        if (
            mapped is not None
            and mapped not in output
        ):
            output.append(
                mapped
            )

    return output


def extract_ucell_column(
    result_adata: ad.AnnData,
    signature: str,
) -> np.ndarray:
    candidates = [
        f"{signature}_UCell",
        f"{signature}_resolvi_UCell",
        signature,
    ]
    source = next(
        (
            column
            for column in candidates
            if column
            in result_adata.obs.columns
        ),
        None,
    )
    if source is None:
        raise KeyError(
            "Could not locate pyUCell output "
            f"for {signature!r}. Last columns: "
            f"{result_adata.obs.columns[-20:].tolist()}"
        )

    return (
        pd.to_numeric(
            result_adata.obs[
                source
            ],
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )


def run_ucell_compat(
    temp: ad.AnnData,
    signatures: dict[
        str,
        list[str],
    ],
    *,
    prefer_device: str,
) -> tuple[
    ad.AnnData,
    dict,
]:
    if not PYUCELL_AVAILABLE:
        raise RuntimeError(
            "pyUCell is required for the expanded tumor signatures."
        )

    candidate_kwargs = {
        "signatures": signatures,
        "layer": None,
        "max_rank": min(
            int(
                UCELL_MAX_RANK
            ),
            temp.n_vars,
        ),
        "ties_method": (
            "min"
            if prefer_device
            != "cpu"
            else "average"
        ),
        "missing_genes": "skip",
        "chunk_size": min(
            int(
                UCELL_INNER_CHUNK_SIZE
            ),
            max(
                1,
                temp.n_obs,
            ),
        ),
        "w_neg": 1.0,
        "suffix": "_UCell",
        "n_jobs": (
            1
            if prefer_device
            != "cpu"
            else int(
                UCELL_CPU_N_JOBS
            )
        ),
        "device": prefer_device,
    }

    kwargs = {
        key: value
        for key, value
        in candidate_kwargs.items()
        if _ucell_accepts(
            key
        )
    }
    omitted = sorted(
        set(
            candidate_kwargs
        )
        - set(
            kwargs
        )
    )

    try:
        result = (
            uc.compute_ucell_scores(
                temp,
                **kwargs,
            )
        )
        if isinstance(
            result,
            ad.AnnData,
        ):
            temp = result

        return (
            temp,
            {
                "backend": (
                    f"device:{prefer_device}"
                    if "device" in kwargs
                    else "legacy_cpu_api"
                ),
                "kwargs_omitted": (
                    omitted
                ),
            },
        )

    except Exception as first_exc:
        if (
            prefer_device
            == "cpu"
            or not
            UCELL_GPU_FALLBACK_TO_CPU
        ):
            raise

        warnings.warn(
            "GPU pyUCell scoring failed; "
            "retrying this chunk on CPU. "
            f"{type(first_exc).__name__}: {first_exc}"
        )

        cpu_kwargs = dict(
            kwargs
        )
        cpu_kwargs.pop(
            "device",
            None,
        )
        if _ucell_accepts(
            "device"
        ):
            cpu_kwargs[
                "device"
            ] = "cpu"
        if (
            "n_jobs"
            in cpu_kwargs
        ):
            cpu_kwargs[
                "n_jobs"
            ] = int(
                UCELL_CPU_N_JOBS
            )
        if (
            "ties_method"
            in cpu_kwargs
        ):
            cpu_kwargs[
                "ties_method"
            ] = "average"

        result = (
            uc.compute_ucell_scores(
                temp,
                **cpu_kwargs,
            )
        )
        if isinstance(
            result,
            ad.AnnData,
        ):
            temp = result

        return (
            temp,
            {
                "backend": (
                    "cpu_fallback"
                ),
                "GPU_error": (
                    f"{type(first_exc).__name__}: "
                    f"{first_exc}"
                ),
                "kwargs_omitted": (
                    omitted
                ),
            },
        )

In [7]:
# ---------------------------------------------------------------------
# Expanded tumor scoring and resume
# ---------------------------------------------------------------------
def open_corrected_layer(
    zarr_path: Path,
):
    root = zarr.open_group(
        str(
            zarr_path
        ),
        mode="r",
    )
    return (
        root[
            "layers"
        ][
            CORRECTED_LAYER
        ]
    )


def score_column_name(
    signature: str,
) -> str:
    return (
        f"tumor_rescue_{signature}_UCell"
    )


def restore_score_staging(
    adata: ad.AnnData,
    path: Path,
    expected_columns: list[str],
) -> bool:
    if not path.exists():
        return False

    staging = pd.read_parquet(
        path
    )
    if "cell_id" not in staging.columns:
        return False

    staging[
        "cell_id"
    ] = (
        staging[
            "cell_id"
        ]
        .astype(str)
    )
    if staging[
        "cell_id"
    ].duplicated().any():
        return False

    staging = staging.set_index(
        "cell_id"
    )
    if len(
        adata.obs_names.difference(
            staging.index
        )
    ):
        return False

    staging = staging.reindex(
        adata.obs_names
    )
    if not set(
        expected_columns
    ).issubset(
        staging.columns
    ):
        return False

    for column in expected_columns:
        adata.obs[
            column
        ] = (
            pd.to_numeric(
                staging[
                    column
                ],
                errors="coerce",
            )
            .fillna(0)
            .to_numpy(
                dtype=np.float32
            )
        )

    print(
        "Restored expanded tumor UCell scores from:",
        path,
    )
    return True


def ensure_expanded_tumor_scores(
    sample: str,
    adata: ad.AnnData,
    paths: dict[
        str,
        Path,
    ],
) -> tuple[
    pd.DataFrame,
    dict,
]:
    cancer_type = (
        SAMPLE_INFO[
            sample
        ][
            "cancer_type"
        ]
    )
    requested = (
        SIGNATURES_BY_CANCER[
            cancer_type
        ]
    )

    present_signatures = {
        signature: map_genes(
            adata.var_names,
            genes,
        )
        for signature, genes
        in requested.items()
    }

    coverage = pd.DataFrame(
        [
            {
                "sample": sample,
                "cancer_type": (
                    cancer_type
                ),
                "signature": (
                    signature
                ),
                "n_requested": len(
                    requested[
                        signature
                    ]
                ),
                "n_present": len(
                    present
                ),
                "fraction_present": (
                    len(
                        present
                    )
                    / max(
                        len(
                            requested[
                                signature
                            ]
                        ),
                        1,
                    )
                ),
                "present_genes": ";".join(
                    present
                ),
            }
            for signature, present
            in present_signatures.items()
        ]
    )
    coverage.to_csv(
        paths[
            "coverage"
        ],
        index=False,
    )

    nonempty = {
        signature: genes
        for signature, genes
        in present_signatures.items()
        if genes
    }
    if not nonempty:
        raise ValueError(
            f"{sample}: no expanded tumor signature genes are present."
        )

    expected_columns = [
        score_column_name(
            signature
        )
        for signature in nonempty
    ]

    already_present = (
        not
        OVERWRITE_EXPANDED_TUMOR_SCORES
        and set(
            expected_columns
        ).issubset(
            adata.obs.columns
        )
    )
    if already_present:
        return (
            coverage,
            {
                "score_source": (
                    "existing_H5AD_columns"
                ),
                "signatures": list(
                    nonempty
                ),
            },
        )

    if (
        RESUME_FROM_SCORE_PARQUET
        and restore_score_staging(
            adata,
            paths[
                "score_staging"
            ],
            expected_columns,
        )
    ):
        return (
            coverage,
            {
                "score_source": (
                    "restored_score_staging"
                ),
                "signatures": list(
                    nonempty
                ),
            },
        )

    if not paths[
        "zarr"
    ].exists():
        raise FileNotFoundError(
            paths[
                "zarr"
            ]
        )

    corrected = open_corrected_layer(
        paths[
            "zarr"
        ]
    )

    if tuple(
        corrected.shape
    ) != tuple(
        adata.shape
    ):
        raise ValueError(
            f"{sample}: corrected Zarr shape "
            f"{corrected.shape} does not match H5AD {adata.shape}."
        )

    if paths[
        "zarr_summary"
    ].exists():
        zarr_summary = json.loads(
            paths[
                "zarr_summary"
            ].read_text(
                encoding="utf-8"
            )
        )
        expected_obs = (
            zarr_summary.get(
                "obs_names_sha256"
            )
        )
        expected_var = (
            zarr_summary.get(
                "var_names_sha256"
            )
        )
        if (
            expected_obs
            and expected_obs
            != names_hash(
                adata.obs_names
            )
        ):
            raise ValueError(
                f"{sample}: H5AD/Zarr obs names do not align."
            )
        if (
            expected_var
            and expected_var
            != names_hash(
                adata.var_names
            )
        ):
            raise ValueError(
                f"{sample}: H5AD/Zarr var names do not align."
            )

    output_arrays = {
        signature: np.zeros(
            adata.n_obs,
            dtype=np.float32,
        )
        for signature in nonempty
    }
    backend_records = []

    for start in range(
        0,
        adata.n_obs,
        int(
            UCELL_OUTER_CELL_CHUNK
        ),
    ):
        end = min(
            start
            + int(
                UCELL_OUTER_CELL_CHUNK
            ),
            adata.n_obs,
        )

        corrected_chunk = np.asarray(
            corrected[
                start:end,
                :,
            ],
            dtype=np.float32,
        )

        temp = ad.AnnData(
            X=corrected_chunk,
            obs=pd.DataFrame(
                index=(
                    adata.obs_names[
                        start:end
                    ].copy()
                )
            ),
            var=pd.DataFrame(
                index=(
                    adata.var_names.copy()
                )
            ),
        )

        temp, backend = run_ucell_compat(
            temp,
            nonempty,
            prefer_device=(
                PREFER_DEVICE
            ),
        )
        backend_records.append(
            backend
        )

        for signature in nonempty:
            output_arrays[
                signature
            ][
                start:end
            ] = extract_ucell_column(
                temp,
                signature,
            )

        del (
            temp,
            corrected_chunk,
        )
        gc.collect()

        print(
            f"{sample}: expanded tumor UCell "
            f"{end:,}/{adata.n_obs:,}"
        )

    staging = pd.DataFrame(
        {
            "cell_id": (
                adata.obs_names.astype(str)
            )
        }
    )
    for signature, values in (
        output_arrays.items()
    ):
        column = score_column_name(
            signature
        )
        adata.obs[
            column
        ] = values
        staging[
            column
        ] = values

    staging.to_parquet(
        paths[
            "score_staging"
        ],
        index=False,
    )

    return (
        coverage,
        {
            "score_source": (
                "fresh_corrected_UCell"
            ),
            "signatures": list(
                nonempty
            ),
            "backend_records_json": json.dumps(
                backend_records[:3],
                default=str,
                sort_keys=True,
            ),
        },
    )

In [8]:
# ---------------------------------------------------------------------
# Raw and reference support
# ---------------------------------------------------------------------
def raw_gene_detection_count(
    adata: ad.AnnData,
    genes: list[str],
) -> tuple[
    np.ndarray,
    dict[str, str | None],
]:
    lookup = {
        str(
            name
        ).upper(): str(
            name
        )
        for name in adata.var_names
    }
    mapping = {
        gene: lookup.get(
            gene.upper()
        )
        for gene in genes
    }
    present = [
        mapped
        for mapped in (
            mapping.values()
        )
        if mapped is not None
    ]

    if not present:
        return (
            np.zeros(
                adata.n_obs,
                dtype=np.int16,
            ),
            mapping,
        )

    matrix = sp.csr_matrix(
        adata[
            :,
            present,
        ].X
    )
    count = np.asarray(
        (
            matrix > 0
        ).sum(
            axis=1
        )
    ).ravel().astype(
        np.int16
    )

    return (
        count,
        mapping,
    )


def combine_text_columns(
    obs: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    columns = [
        column
        for column in columns
        if column in obs.columns
    ]
    if not columns:
        return pd.Series(
            "",
            index=(
                obs.index
            ),
            dtype="string",
        )

    output = (
        obs[
            columns[0]
        ]
        .astype("string")
        .fillna("")
    )
    for column in columns[1:]:
        output = (
            output
            + " | "
            + obs[
                column
            ]
            .astype("string")
            .fillna("")
        )

    return output.str.lower()


def epithelial_reference_support(
    adata: ad.AnnData,
) -> np.ndarray:
    if not USE_REFERENCE_SUPPORT:
        return np.zeros(
            adata.n_obs,
            dtype=bool,
        )

    text = combine_text_columns(
        adata.obs,
        [
            "full_hierarchical_labels",
            "final_level_labels",
            "azimuth_broad",
            "azimuth_medium",
            "azimuth_fine",
            "azimuth_broad_CL",
            "azimuth_fine_CL",
        ],
    )

    pattern = (
        r"\b("
        r"epithelial|epithelium|tumou?r|malignant|cancer|"
        r"carcinoma|melanoma|melanocyte|keratinocyte|"
        r"basal|alveolar|club|ciliated|goblet|"
        r"enterocyte|colonocyte|secretory"
        r")\b"
    )

    return (
        text.str.contains(
            pattern,
            regex=True,
            na=False,
        )
        .to_numpy(
            dtype=bool
        )
    )

In [9]:
# ---------------------------------------------------------------------
# Percentiles and broader tumor/epithelial rescue
# ---------------------------------------------------------------------
def percentile_rank(
    values: np.ndarray,
) -> np.ndarray:
    values = np.asarray(
        values,
        dtype=float,
    )
    finite = np.isfinite(
        values
    )

    output = np.full(
        len(
            values
        ),
        np.nan,
        dtype=np.float32,
    )
    if finite.sum() == 0:
        return output

    output[
        finite
    ] = (
        pd.Series(
            values[
                finite
            ]
        )
        .rank(
            method="average",
            pct=True,
        )
        .to_numpy(
            dtype=np.float32
        )
    )

    return output


def quantile_floor_threshold(
    values: np.ndarray,
    *,
    quantile: float,
    floor: float,
) -> dict:
    values = np.asarray(
        values,
        dtype=float,
    )
    finite = np.isfinite(
        values
    )

    if finite.sum() == 0:
        threshold = float(
            "inf"
        )
        quantile_cut = float(
            "nan"
        )
    else:
        quantile_cut = float(
            np.quantile(
                values[
                    finite
                ],
                float(
                    quantile
                ),
            )
        )
        threshold = max(
            float(
                floor
            ),
            quantile_cut,
        )

    return {
        "threshold": threshold,
        "quantile_cut": (
            quantile_cut
        ),
        "strong": (
            finite
            & (
                values
                >= threshold
            )
        ),
    }


def numeric_obs(
    adata: ad.AnnData,
    column: str,
) -> np.ndarray:
    if column not in adata.obs.columns:
        return np.zeros(
            adata.n_obs,
            dtype=np.float32,
        )

    return (
        pd.to_numeric(
            adata.obs[
                column
            ],
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )


def apply_broader_tumor_rescue(
    adata: ad.AnnData,
    sample: str,
) -> tuple[
    pd.DataFrame,
    dict,
]:
    cancer_type = (
        SAMPLE_INFO[
            sample
        ][
            "cancer_type"
        ]
    )
    signatures = (
        SIGNATURES_BY_CANCER[
            cancer_type
        ]
    )

    score_columns = {
        signature: score_column_name(
            signature
        )
        for signature in signatures
        if score_column_name(
            signature
        )
        in adata.obs.columns
    }
    if not score_columns:
        raise KeyError(
            f"{sample}: no expanded tumor score columns are present."
        )

    score_matrix = np.column_stack(
        [
            numeric_obs(
                adata,
                column,
            )
            for column in (
                score_columns.values()
            )
        ]
    )
    score_names = list(
        score_columns
    )

    # Melanoma invasive score can drive rescue only with lineage/reference
    # support. It is temporarily retained here and masked below.
    max_index = np.argmax(
        score_matrix,
        axis=1,
    )
    expanded_score = score_matrix[
        np.arange(
            adata.n_obs
        ),
        max_index,
    ]
    dominant_program = np.asarray(
        [
            score_names[
                index
            ]
            for index in (
                max_index
            )
        ],
        dtype=object,
    )

    reference_support = (
        epithelial_reference_support(
            adata
        )
    )

    all_relevant_raw_genes = list(
        dict.fromkeys(
            gene
            for genes in (
                signatures.values()
            )
            for gene in genes
        )
    )
    raw_gene_count, raw_mapping = (
        raw_gene_detection_count(
            adata,
            all_relevant_raw_genes,
        )
    )

    high_specificity_count, high_specificity_mapping = (
        raw_gene_detection_count(
            adata,
            HIGH_SPECIFICITY_MARKERS[
                cancer_type
            ],
        )
    )
    high_specificity_support = (
        high_specificity_count
        >= 1
    )

    if cancer_type == "melanoma":
        melanoma_lineage_column = (
            score_column_name(
                "Melanoma_lineage_expanded"
            )
        )
        melanoma_lineage_score = (
            numeric_obs(
                adata,
                melanoma_lineage_column,
            )
        )
        melanoma_lineage_percentile = (
            percentile_rank(
                melanoma_lineage_score
            )
        )
        melanoma_lineage_raw_count, _ = (
            raw_gene_detection_count(
                adata,
                MELANOMA_LINEAGE_EXPANDED,
            )
        )
        invasive_driver_allowed = (
            (
                melanoma_lineage_percentile
                >= 0.60
            )
            | (
                melanoma_lineage_raw_count
                >= 1
            )
            | reference_support
        )

        invasive_is_max = (
            dominant_program
            == "Melanoma_invasive_support"
        )
        disallowed = (
            invasive_is_max
            & ~invasive_driver_allowed
        )

        if disallowed.any():
            alternative_names = [
                name
                for name in score_names
                if name
                != "Melanoma_invasive_support"
            ]
            alternative_matrix = np.column_stack(
                [
                    numeric_obs(
                        adata,
                        score_columns[
                            name
                        ],
                    )
                    for name in alternative_names
                ]
            )
            alternative_index = np.argmax(
                alternative_matrix[
                    disallowed
                ],
                axis=1,
            )
            expanded_score[
                disallowed
            ] = alternative_matrix[
                disallowed,
                alternative_index,
            ]
            dominant_program[
                disallowed
            ] = np.asarray(
                alternative_names,
                dtype=object,
            )[
                alternative_index
            ]

    expanded_percentile = (
        percentile_rank(
            expanded_score
        )
    )

    competing_percentiles = np.column_stack(
        [
            percentile_rank(
                numeric_obs(
                    adata,
                    column,
                )
            )
            for column in (
                COMPETING_SCORE_COLUMNS.values()
            )
        ]
    )
    max_competing_percentile = np.nanmax(
        competing_percentiles,
        axis=1,
    )

    primary_threshold = quantile_floor_threshold(
        expanded_score,
        quantile=(
            PRIMARY_TUMOR_QUANTILE
        ),
        floor=(
            PRIMARY_SCORE_FLOOR
        ),
    )
    exploratory_threshold = quantile_floor_threshold(
        expanded_score,
        quantile=(
            EXPLORATORY_TUMOR_QUANTILE
        ),
        floor=(
            EXPLORATORY_SCORE_FLOOR
        ),
    )

    current_label = (
        adata.obs[
            "prelim_cell_type_primary"
        ]
        .astype(str)
        .to_numpy(
            dtype=object
        )
    )
    existing_tumor = (
        current_label
        == "Tumor"
    )
    eligible = (
        current_label
        == ELIGIBLE_CURRENT_LABEL
    )

    primary_raw_support = (
        raw_gene_count
        >= int(
            PRIMARY_MIN_RAW_GENES
        )
    )
    exploratory_raw_support = (
        raw_gene_count
        >= int(
            EXPLORATORY_MIN_RAW_GENES
        )
    )

    if ALLOW_ONE_HIGH_SPECIFICITY_RAW_MARKER:
        primary_raw_support |= (
            high_specificity_support
        )
        exploratory_raw_support |= (
            high_specificity_support
        )

    if USE_REFERENCE_SUPPORT:
        primary_support = (
            primary_raw_support
            | reference_support
        )
        exploratory_support = (
            exploratory_raw_support
            | reference_support
        )
    else:
        primary_support = (
            primary_raw_support
        )
        exploratory_support = (
            exploratory_raw_support
        )

    primary_rescue = (
        eligible
        & primary_threshold[
            "strong"
        ]
        & primary_support
        & (
            expanded_percentile
            >= (
                max_competing_percentile
                - float(
                    PRIMARY_COMPETITION_ALLOWANCE
                )
            )
        )
    )

    exploratory_rescue = (
        eligible
        & ~primary_rescue
        & exploratory_threshold[
            "strong"
        ]
        & exploratory_support
        & (
            expanded_percentile
            >= (
                max_competing_percentile
                - float(
                    EXPLORATORY_COMPETITION_ALLOWANCE
                )
            )
        )
    )

    tier = np.full(
        adata.n_obs,
        "not_tumor",
        dtype=object,
    )
    tier[
        exploratory_rescue
    ] = "exploratory_only"
    tier[
        primary_rescue
    ] = "primary_rescue"
    tier[
        existing_tumor
    ] = "existing_tumor"

    rank = np.zeros(
        adata.n_obs,
        dtype=np.int8,
    )
    rank[
        exploratory_rescue
    ] = 1
    rank[
        primary_rescue
    ] = 2
    rank[
        existing_tumor
    ] = 3

    primary_annotation = (
        current_label.copy()
    )
    primary_annotation[
        existing_tumor
        | primary_rescue
    ] = EXPANDED_TUMOR_LABEL

    exploratory_annotation = (
        current_label.copy()
    )
    exploratory_annotation[
        existing_tumor
        | primary_rescue
        | exploratory_rescue
    ] = EXPANDED_TUMOR_LABEL

    adata.obs[
        "tumor_epithelial_expanded_score"
    ] = expanded_score.astype(
        np.float32
    )
    adata.obs[
        "tumor_epithelial_expanded_percentile"
    ] = expanded_percentile.astype(
        np.float32
    )
    adata.obs[
        "tumor_epithelial_minus_max_competing_percentile"
    ] = (
        expanded_percentile
        - max_competing_percentile
    ).astype(
        np.float32
    )
    adata.obs[
        "tumor_epithelial_dominant_program"
    ] = pd.Categorical(
        dominant_program
    )
    adata.obs[
        "tumor_epithelial_raw_gene_count"
    ] = raw_gene_count
    adata.obs[
        "tumor_epithelial_high_specificity_raw_support"
    ] = (
        high_specificity_support
    )
    adata.obs[
        "tumor_epithelial_reference_support"
    ] = reference_support
    adata.obs[
        "tumor_epithelial_confidence_tier"
    ] = pd.Categorical(
        tier,
        categories=[
            "existing_tumor",
            "primary_rescue",
            "exploratory_only",
            "not_tumor",
        ],
        ordered=True,
    )
    adata.obs[
        "tumor_epithelial_confidence_rank"
    ] = rank
    adata.obs[
        "tumor_epithelial_primary"
    ] = (
        existing_tumor
        | primary_rescue
    )
    adata.obs[
        "tumor_epithelial_exploratory"
    ] = (
        existing_tumor
        | primary_rescue
        | exploratory_rescue
    )

    adata.obs[
        "prelim_cell_type_primary_tumor_expanded"
    ] = pd.Categorical(
        primary_annotation
    )
    adata.obs[
        "prelim_cell_type_exploratory_tumor_expanded"
    ] = pd.Categorical(
        exploratory_annotation
    )
    adata.obs[
        "cell_type_preliminary_tumor_expanded"
    ] = adata.obs[
        "prelim_cell_type_primary_tumor_expanded"
    ].copy()

    thresholds = pd.DataFrame(
        [
            {
                "sample": sample,
                "cancer_type": (
                    cancer_type
                ),
                "tier": (
                    "primary"
                ),
                "quantile": (
                    PRIMARY_TUMOR_QUANTILE
                ),
                "quantile_cut": (
                    primary_threshold[
                        "quantile_cut"
                    ]
                ),
                "fixed_floor": (
                    PRIMARY_SCORE_FLOOR
                ),
                "final_threshold": (
                    primary_threshold[
                        "threshold"
                    ]
                ),
                "competition_allowance": (
                    PRIMARY_COMPETITION_ALLOWANCE
                ),
                "minimum_raw_genes": (
                    PRIMARY_MIN_RAW_GENES
                ),
                "n_score_strong": int(
                    primary_threshold[
                        "strong"
                    ].sum()
                ),
                "n_rescued": int(
                    primary_rescue.sum()
                ),
            },
            {
                "sample": sample,
                "cancer_type": (
                    cancer_type
                ),
                "tier": (
                    "exploratory"
                ),
                "quantile": (
                    EXPLORATORY_TUMOR_QUANTILE
                ),
                "quantile_cut": (
                    exploratory_threshold[
                        "quantile_cut"
                    ]
                ),
                "fixed_floor": (
                    EXPLORATORY_SCORE_FLOOR
                ),
                "final_threshold": (
                    exploratory_threshold[
                        "threshold"
                    ]
                ),
                "competition_allowance": (
                    EXPLORATORY_COMPETITION_ALLOWANCE
                ),
                "minimum_raw_genes": (
                    EXPLORATORY_MIN_RAW_GENES
                ),
                "n_score_strong": int(
                    exploratory_threshold[
                        "strong"
                    ].sum()
                ),
                "n_rescued": int(
                    exploratory_rescue.sum()
                ),
            },
        ]
    )

    summary = {
        "n_cells": int(
            adata.n_obs
        ),
        "n_existing_tumor": int(
            existing_tumor.sum()
        ),
        "n_primary_rescued_from_unresolved": int(
            primary_rescue.sum()
        ),
        "n_exploratory_rescued_from_unresolved": int(
            exploratory_rescue.sum()
        ),
        "n_primary_tumor_epithelial_total": int(
            (
                existing_tumor
                | primary_rescue
            ).sum()
        ),
        "n_exploratory_tumor_epithelial_total": int(
            (
                existing_tumor
                | primary_rescue
                | exploratory_rescue
            ).sum()
        ),
        "n_unresolved_before": int(
            eligible.sum()
        ),
        "n_unresolved_after_primary": int(
            (
                eligible
                & ~primary_rescue
            ).sum()
        ),
        "n_unresolved_after_exploratory": int(
            (
                eligible
                & ~primary_rescue
                & ~exploratory_rescue
            ).sum()
        ),
        "raw_gene_mapping_json": json.dumps(
            raw_mapping,
            default=str,
            sort_keys=True,
        ),
        "high_specificity_mapping_json": json.dumps(
            high_specificity_mapping,
            default=str,
            sort_keys=True,
        ),
    }

    return (
        thresholds,
        summary,
    )

In [10]:
# ---------------------------------------------------------------------
# Diagnostic plots
# ---------------------------------------------------------------------
def plotting_indices(
    n_obs: int,
) -> np.ndarray:
    if n_obs <= int(
        PLOT_MAX_CELLS
    ):
        return np.arange(
            n_obs
        )

    rng = np.random.default_rng(
        RANDOM_STATE
    )
    return np.sort(
        rng.choice(
            n_obs,
            size=int(
                PLOT_MAX_CELLS
            ),
            replace=False,
        )
    )


def save_score_audit_plot(
    adata: ad.AnnData,
    thresholds: pd.DataFrame,
    sample: str,
    path: Path,
) -> None:
    score = numeric_obs(
        adata,
        "tumor_epithelial_expanded_score",
    )
    percentile = numeric_obs(
        adata,
        "tumor_epithelial_expanded_percentile",
    )
    margin = numeric_obs(
        adata,
        "tumor_epithelial_minus_max_competing_percentile",
    )
    tier = (
        adata.obs[
            "tumor_epithelial_confidence_tier"
        ]
        .astype(str)
        .to_numpy()
    )
    current = (
        adata.obs[
            "prelim_cell_type_primary"
        ]
        .astype(str)
        .to_numpy()
    )

    primary_threshold = float(
        thresholds.loc[
            thresholds[
                "tier"
            ]
            == "primary",
            "final_threshold",
        ].iloc[0]
    )
    exploratory_threshold = float(
        thresholds.loc[
            thresholds[
                "tier"
            ]
            == "exploratory",
            "final_threshold",
        ].iloc[0]
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
    )

    unresolved = (
        current
        == ELIGIBLE_CURRENT_LABEL
    )
    existing_tumor = (
        current
        == "Tumor"
    )

    axes[0].hist(
        score[
            unresolved
        ],
        bins=80,
        alpha=0.65,
        label="current unresolved",
        log=True,
    )
    if existing_tumor.any():
        axes[0].hist(
            score[
                existing_tumor
            ],
            bins=80,
            alpha=0.55,
            label="existing tumor",
            log=True,
        )
    axes[0].axvline(
        primary_threshold,
        linestyle="-",
        linewidth=2,
        label=(
            f"primary threshold: "
            f"{primary_threshold:.4g}"
        ),
    )
    axes[0].axvline(
        exploratory_threshold,
        linestyle="--",
        linewidth=1.5,
        label=(
            f"exploratory threshold: "
            f"{exploratory_threshold:.4g}"
        ),
    )
    axes[0].set_xlabel(
        "Expanded tumor/epithelial UCell score"
    )
    axes[0].set_ylabel(
        "Cells"
    )
    axes[0].set_title(
        "Score distributions and thresholds"
    )
    axes[0].legend(
        fontsize=8,
        frameon=False,
    )

    indices = plotting_indices(
        adata.n_obs
    )
    axes[1].scatter(
        (
            percentile[
                indices
            ]
            - margin[
                indices
            ]
        ),
        percentile[
            indices
        ],
        s=4,
        linewidths=0,
        alpha=0.20,
        rasterized=True,
    )

    for label in (
        "existing_tumor",
        "primary_rescue",
        "exploratory_only",
    ):
        mask = (
            tier
            == label
        )
        if mask.any():
            axes[1].scatter(
                (
                    percentile[
                        mask
                    ]
                    - margin[
                        mask
                    ]
                ),
                percentile[
                    mask
                ],
                s=9,
                linewidths=0,
                alpha=0.80,
                label=label,
                rasterized=True,
            )

    x = np.linspace(
        0,
        1,
        200,
    )
    axes[1].plot(
        x,
        x
        - float(
            PRIMARY_COMPETITION_ALLOWANCE
        ),
        linestyle="-",
        label="primary competition allowance",
    )
    axes[1].plot(
        x,
        x
        - float(
            EXPLORATORY_COMPETITION_ALLOWANCE
        ),
        linestyle="--",
        label="exploratory competition allowance",
    )
    axes[1].set_xlabel(
        "Maximum competing lineage percentile"
    )
    axes[1].set_ylabel(
        "Expanded tumor/epithelial percentile"
    )
    axes[1].set_title(
        "Tumor score versus competing lineages"
    )
    axes[1].legend(
        fontsize=7,
        frameon=False,
    )

    fig.suptitle(
        f"{sample}: broader tumor/epithelial rescue"
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_spatial_rescue_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> str | None:
    key = choose_spatial_key(
        adata
    )
    if key is None:
        return None

    coordinates = np.asarray(
        adata.obsm[
            key
        ],
        dtype=float,
    )
    indices = plotting_indices(
        adata.n_obs
    )

    tier = (
        adata.obs[
            "tumor_epithelial_confidence_tier"
        ]
        .astype(str)
        .to_numpy()
    )
    plot_tier = tier[
        indices
    ]

    categories = [
        category
        for category in (
            "not_tumor",
            "exploratory_only",
            "primary_rescue",
            "existing_tumor",
        )
        if (
            plot_tier
            == category
        ).any()
    ]

    fig, ax = plt.subplots(
        figsize=(9, 8),
    )

    for category in categories:
        mask = (
            plot_tier
            == category
        )
        ax.scatter(
            coordinates[
                indices[
                    mask
                ],
                0,
            ],
            coordinates[
                indices[
                    mask
                ],
                1,
            ],
            s=(
                4
                if category
                != "not_tumor"
                else 1
            ),
            linewidths=0,
            alpha=(
                0.85
                if category
                != "not_tumor"
                else 0.20
            ),
            label=category,
            rasterized=True,
        )

    ax.set_title(
        f"{sample}: existing and rescued tumor/epithelial cells"
    )
    ax.set_xlabel(
        "Spatial x"
    )
    ax.set_ylabel(
        "Spatial y"
    )
    ax.set_aspect(
        "equal"
    )
    ax.invert_yaxis()
    ax.legend(
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=8,
        frameon=False,
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    return key

In [11]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(
    sample: str,
) -> tuple[
    dict,
    pd.DataFrame,
]:
    paths = paths_for_sample(
        sample
    )

    if not paths[
        "source"
    ].exists():
        raise FileNotFoundError(
            paths[
                "source"
            ]
        )

    print(
        "\n"
        + "=" * 90
    )
    print(
        "Broader tumor/epithelial rescue:",
        sample,
    )
    print(
        "Input:",
        paths[
            "source"
        ],
    )

    adata = ad.read_h5ad(
        paths[
            "source"
        ]
    )
    adata.obs_names = (
        adata.obs_names.astype(str)
    )

    for column, value in (
        SAMPLE_INFO[
            sample
        ].items()
    ):
        adata.obs[
            column
        ] = value
    adata.obs[
        "sample"
    ] = sample

    coverage, scoring_info = (
        ensure_expanded_tumor_scores(
            sample,
            adata,
            paths,
        )
    )

    thresholds, rescue_summary = (
        apply_broader_tumor_rescue(
            adata,
            sample,
        )
    )
    thresholds.to_csv(
        paths[
            "thresholds"
        ],
        index=False,
    )

    old_labels = (
        adata.obs[
            "prelim_cell_type_primary"
        ]
        .astype(str)
    )
    primary_labels = (
        adata.obs[
            "prelim_cell_type_primary_tumor_expanded"
        ]
        .astype(str)
    )
    exploratory_labels = (
        adata.obs[
            "prelim_cell_type_exploratory_tumor_expanded"
        ]
        .astype(str)
    )

    before = (
        old_labels
        .value_counts()
        .rename(
            "before_n_cells"
        )
    )
    after_primary = (
        primary_labels
        .value_counts()
        .rename(
            "after_primary_n_cells"
        )
    )
    after_exploratory = (
        exploratory_labels
        .value_counts()
        .rename(
            "after_exploratory_n_cells"
        )
    )

    before_after = pd.concat(
        [
            before,
            after_primary,
            after_exploratory,
        ],
        axis=1,
    ).fillna(
        0
    ).astype(
        int
    )
    before_after[
        "sample"
    ] = sample
    before_after[
        "patient"
    ] = (
        SAMPLE_INFO[
            sample
        ][
            "patient"
        ]
    )
    before_after[
        "cancer_type"
    ] = (
        SAMPLE_INFO[
            sample
        ][
            "cancer_type"
        ]
    )
    before_after[
        "biopsy_stage"
    ] = (
        SAMPLE_INFO[
            sample
        ][
            "biopsy_stage"
        ]
    )
    before_after = (
        before_after
        .rename_axis(
            "cell_type"
        )
        .reset_index()
    )
    before_after.to_csv(
        paths[
            "counts"
        ],
        index=False,
    )

    rescued_ids = pd.DataFrame(
        {
            "cell_id": (
                adata.obs_names.astype(str)
            ),
            "current_cell_type": (
                old_labels.to_numpy()
            ),
            "primary_expanded_cell_type": (
                primary_labels.to_numpy()
            ),
            "exploratory_expanded_cell_type": (
                exploratory_labels.to_numpy()
            ),
            "tumor_epithelial_confidence_tier": (
                adata.obs[
                    "tumor_epithelial_confidence_tier"
                ]
                .astype(str)
                .to_numpy()
            ),
            "tumor_epithelial_dominant_program": (
                adata.obs[
                    "tumor_epithelial_dominant_program"
                ]
                .astype(str)
                .to_numpy()
            ),
            "tumor_epithelial_expanded_score": (
                adata.obs[
                    "tumor_epithelial_expanded_score"
                ].to_numpy(
                    dtype=np.float32
                )
            ),
            "tumor_epithelial_expanded_percentile": (
                adata.obs[
                    "tumor_epithelial_expanded_percentile"
                ].to_numpy(
                    dtype=np.float32
                )
            ),
            "tumor_epithelial_raw_gene_count": (
                adata.obs[
                    "tumor_epithelial_raw_gene_count"
                ].to_numpy()
            ),
            "tumor_epithelial_high_specificity_raw_support": (
                adata.obs[
                    "tumor_epithelial_high_specificity_raw_support"
                ].to_numpy(
                    dtype=bool
                )
            ),
            "tumor_epithelial_reference_support": (
                adata.obs[
                    "tumor_epithelial_reference_support"
                ].to_numpy(
                    dtype=bool
                )
            ),
        }
    )

    if WRITE_RESCUED_CELL_IDS:
        rescued_ids.to_csv(
            paths[
                "rescued_ids"
            ],
            index=False,
        )

    output_columns = [
        column
        for column in (
            adata.obs.columns
        )
        if (
            column.startswith(
                "tumor_"
            )
            or column.startswith(
                "prelim_"
            )
            or column.startswith(
                "rescue_"
            )
            or column.startswith(
                "fallback_"
            )
            or column
            in {
                "cell_type_preliminary",
                "cell_type_preliminary_tumor_expanded",
                "T_confidence",
                "Treg_confidence",
                "sample",
                "patient",
                "cancer_type",
                "biopsy_stage",
            }
        )
    ]
    metadata = (
        adata.obs[
            output_columns
        ]
        .copy()
    )
    metadata.insert(
        0,
        "cell_id",
        adata.obs_names.astype(str),
    )

    if WRITE_METADATA_PARQUET:
        metadata.to_parquet(
            paths[
                "metadata"
            ],
            index=False,
        )

    spatial_key = choose_spatial_key(
        adata
    )
    if (
        WRITE_SPATIAL_PARQUET
        and spatial_key
        is not None
    ):
        coordinates = np.asarray(
            adata.obsm[
                spatial_key
            ],
            dtype=np.float32,
        )
        spatial = (
            metadata.copy()
        )
        spatial[
            "spatial_x"
        ] = coordinates[
            :,
            0,
        ]
        spatial[
            "spatial_y"
        ] = coordinates[
            :,
            1,
        ]
        spatial[
            "spatial_source"
        ] = spatial_key
        spatial.to_parquet(
            paths[
                "spatial"
            ],
            index=False,
        )

    save_score_audit_plot(
        adata,
        thresholds,
        sample,
        paths[
            "figures"
        ]
        / f"{sample}_expanded_tumor_score_audit.png",
    )
    spatial_key = (
        save_spatial_rescue_plot(
            adata,
            sample,
            paths[
                "figures"
            ]
            / f"{sample}_expanded_tumor_spatial.png",
        )
    )

    adata.uns[
        "broader_tumor_epithelial_rescue"
    ] = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "source_step11_h5ad": str(
            paths[
                "source"
            ]
        ),
        "corrected_zarr": str(
            paths[
                "zarr"
            ]
        ),
        "corrected_layer": (
            CORRECTED_LAYER
        ),
        "primary_label_column": (
            "prelim_cell_type_primary_tumor_expanded"
        ),
        "exploratory_label_column": (
            "prelim_cell_type_exploratory_tumor_expanded"
        ),
        "confidence_column": (
            "tumor_epithelial_confidence_tier"
        ),
        "label_interpretation": (
            "Tumor/epithelial combines malignant and non-malignant "
            "epithelial transcriptional programs."
        ),
        "scoring_info_json": json.dumps(
            scoring_info,
            default=str,
            sort_keys=True,
        ),
    }

    if WRITE_ANNOTATED_H5AD:
        safe_write_h5ad(
            adata,
            paths[
                "annotated"
            ],
            compression=(
                H5AD_COMPRESSION
            ),
        )

    if (
        VALIDATE_WRITTEN_H5AD
        and WRITE_ANNOTATED_H5AD
    ):
        check = ad.read_h5ad(
            paths[
                "annotated"
            ],
            backed="r",
        )
        try:
            if check.n_obs != adata.n_obs:
                raise RuntimeError(
                    f"{sample}: written n_obs changed."
                )
            for column in (
                "prelim_cell_type_primary_tumor_expanded",
                "tumor_epithelial_confidence_tier",
                "tumor_epithelial_dominant_program",
            ):
                if column not in check.obs.columns:
                    raise RuntimeError(
                        f"{sample}: written object is missing {column}."
                    )
        finally:
            check.file.close()

    summary = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "sample": sample,
        **SAMPLE_INFO[
            sample
        ],
        **rescue_summary,
        "scoring_info": (
            scoring_info
        ),
        "spatial_key": (
            spatial_key
        ),
        "annotated_h5ad": str(
            paths[
                "annotated"
            ]
        )
        if WRITE_ANNOTATED_H5AD
        else None,
        "metadata_parquet": str(
            paths[
                "metadata"
            ]
        )
        if WRITE_METADATA_PARQUET
        else None,
    }
    write_json(
        summary,
        paths[
            "summary"
        ],
    )

    print(
        sample,
        {
            "existing_tumor": (
                summary[
                    "n_existing_tumor"
                ]
            ),
            "primary_rescued": (
                summary[
                    "n_primary_rescued_from_unresolved"
                ]
            ),
            "exploratory_rescued": (
                summary[
                    "n_exploratory_rescued_from_unresolved"
                ]
            ),
            "unresolved_before": (
                summary[
                    "n_unresolved_before"
                ]
            ),
            "unresolved_after_primary": (
                summary[
                    "n_unresolved_after_primary"
                ]
            ),
        },
    )

    del adata
    gc.collect()

    return (
        summary,
        before_after,
    )

In [12]:
# ---------------------------------------------------------------------
# Run all samples
# ---------------------------------------------------------------------
results = {}
failures = {}
all_before_after = []

for sample in SECTION_NAMES:
    try:
        summary, before_after = (
            process_sample(
                sample
            )
        )
        results[
            sample
        ] = summary
        all_before_after.append(
            before_after
        )

    except Exception as exc:
        failures[
            sample
        ] = (
            f"{type(exc).__name__}: {exc}"
        )
        print(
            f"[FAILED] {sample}: "
            f"{type(exc).__name__}: {exc}"
        )
        traceback.print_exc(
            limit=12
        )

        if not CONTINUE_ON_ERROR:
            raise

    finally:
        plt.close(
            "all"
        )
        gc.collect()

write_json(
    results,
    OUTPUT_ROOT
    / "all_sample_broader_tumor_results.json",
)
write_json(
    failures,
    OUTPUT_ROOT
    / "all_sample_broader_tumor_failures.json",
)

before_after_long = (
    pd.concat(
        all_before_after,
        ignore_index=True,
    )
    if all_before_after
    else pd.DataFrame()
)
before_after_long.to_csv(
    TABLE_ROOT
    / "before_after_cell_type_counts_all_samples.csv",
    index=False,
)

print(
    "Completed:",
    sorted(
        results
    ),
)
print(
    "Failures:",
    json.dumps(
        failures,
        indent=2,
    ),
)


Broader tumor/epithelial rescue: Screen_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/Screen_39_21/Screen_39_21_preliminary_cell_type_annotated.h5ad
Screen_39_21: expanded tumor UCell 5,000/22,950
Screen_39_21: expanded tumor UCell 10,000/22,950
Screen_39_21: expanded tumor UCell 15,000/22,950
Screen_39_21: expanded tumor UCell 20,000/22,950
Screen_39_21: expanded tumor UCell 22,950/22,950


/tmp/ipykernel_95195/3253919498.py:135: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue/Screen_39_21/Screen_39_21_preliminary_tumor_expanded.h5ad
Screen_39_21 {'existing_tumor': 2083, 'primary_rescued': 1385, 'exploratory_rescued': 954, 'unresolved_before': 13658, 'unresolved_after_primary': 12273}

Broader tumor/epithelial rescue: C2D15_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/C2D15_39_21/C2D15_39_21_preliminary_cell_type_annotated.h5ad
C2D15_39_21: expanded tumor UCell 5,000/10,822
C2D15_39_21: expanded tumor UCell 10,000/10,822
C2D15_39_21: expanded tumor UCell 10,822/10,822


/tmp/ipykernel_95195/3253919498.py:135: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue/C2D15_39_21/C2D15_39_21_preliminary_tumor_expanded.h5ad
C2D15_39_21 {'existing_tumor': 1017, 'primary_rescued': 207, 'exploratory_rescued': 436, 'unresolved_before': 7332, 'unresolved_after_primary': 7125}

Broader tumor/epithelial rescue: Screen_17_26
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/Screen_17_26/Screen_17_26_preliminary_cell_type_annotated.h5ad
Screen_17_26: expanded tumor UCell 5,000/47,896
Screen_17_26: expanded tumor UCell 10,000/47,896
Screen_17_26: expanded tumor UCell 15,000/47,896
Screen_17_26: expanded tumor UCell 20,000/47,896
Screen_17_26: expanded tumor UCell 25,000/47,896
Screen_17_26: expanded tumor UCell 30,000/47,896
Screen_17_26: expanded tumor UCell 35,000/47,896
Screen_17_26: expanded tumor UCell 40,000/47,896
Screen

...

Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue/C2D15_30_16/C2D15_30_16_preliminary_tumor_expanded.h5ad
C2D15_30_16 {'existing_tumor': 3417, 'primary_rescued': 41026, 'exploratory_rescued': 30706, 'unresolved_before': 286392, 'unresolved_after_primary': 245366}

Broader tumor/epithelial rescue: Screen_23_25
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/Screen_23_25/Screen_23_25_preliminary_cell_type_annotated.h5ad
Screen_23_25: expanded tumor UCell 5,000/25,776
Screen_23_25: expanded tumor UCell 10,000/25,776
Screen_23_25: expanded tumor UCell 15,000/25,776
Screen_23_25: expanded tumor UCell 20,000/25,776
Screen_23_25: expanded tumor UCell 25,000/25,776
Screen_23_25: expanded tumor UCell 25,776/25,776


/tmp/ipykernel_95195/3253919498.py:135: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue/Screen_23_25/Screen_23_25_preliminary_tumor_expanded.h5ad
Screen_23_25 {'existing_tumor': 907, 'primary_rescued': 3145, 'exploratory_rescued': 2420, 'unresolved_before': 18419, 'unresolved_after_primary': 15274}

Broader tumor/epithelial rescue: C2D15_23_25
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/11_preliminary_cell_type_annotation/C2D15_23_25/C2D15_23_25_preliminary_cell_type_annotated.h5ad
C2D15_23_25: expanded tumor UCell 5,000/10,243
C2D15_23_25: expanded tumor UCell 10,000/10,243
C2D15_23_25: expanded tumor UCell 10,243/10,243


/tmp/ipykernel_95195/3253919498.py:135: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/12_broader_tumor_epithelial_rescue/C2D15_23_25/C2D15_23_25_preliminary_tumor_expanded.h5ad
C2D15_23_25 {'existing_tumor': 227, 'primary_rescued': 987, 'exploratory_rescued': 882, 'unresolved_before': 7077, 'unresolved_after_primary': 6090}
Completed: ['C2D15_16_22', 'C2D15_17_26', 'C2D15_18_23', 'C2D15_23_25', 'C2D15_30_16', 'C2D15_39_21', 'Screen_16_22', 'Screen_17_26', 'Screen_18_23', 'Screen_23_25', 'Screen_30_16', 'Screen_39_21']
Failures: {}


In [13]:
# ---------------------------------------------------------------------
# Aggregate primary and exploratory composition tables
# ---------------------------------------------------------------------
if not results:
    raise RuntimeError(
        "No samples completed."
    )

summary = pd.DataFrame(
    list(
        results.values()
    )
)
summary.to_csv(
    TABLE_ROOT
    / "broader_tumor_rescue_summary_by_sample.csv",
    index=False,
)

composition_rows = []

for sample in sorted(
    results
):
    paths = paths_for_sample(
        sample
    )
    backed = ad.read_h5ad(
        paths[
            "annotated"
        ],
        backed="r",
    )
    try:
        for annotation_name, column in (
            (
                "primary",
                "prelim_cell_type_primary_tumor_expanded",
            ),
            (
                "exploratory",
                "prelim_cell_type_exploratory_tumor_expanded",
            ),
        ):
            labels = (
                backed.obs[
                    column
                ]
                .astype(str)
            )
            counts = (
                labels
                .value_counts()
            )

            for cell_type, n_cells in (
                counts.items()
            ):
                composition_rows.append(
                    {
                        "sample": sample,
                        **SAMPLE_INFO[
                            sample
                        ],
                        "annotation_tier": (
                            annotation_name
                        ),
                        "cell_type": (
                            cell_type
                        ),
                        "n_cells": int(
                            n_cells
                        ),
                        "percentage": (
                            100.0
                            * float(
                                n_cells
                            )
                            / max(
                                backed.n_obs,
                                1,
                            )
                        ),
                    }
                )
    finally:
        backed.file.close()

composition = pd.DataFrame(
    composition_rows
)
composition.to_csv(
    TABLE_ROOT
    / "expanded_cell_type_counts_and_percentages_by_sample.csv",
    index=False,
)

primary_composition = composition.loc[
    composition[
        "annotation_tier"
    ]
    == "primary"
].copy()
exploratory_composition = composition.loc[
    composition[
        "annotation_tier"
    ]
    == "exploratory"
].copy()

primary_percentage_wide = (
    primary_composition
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        columns="cell_type",
        values="percentage",
        fill_value=0,
        observed=True,
    )
    .sort_index()
)
primary_count_wide = (
    primary_composition
    .pivot_table(
        index=[
            "cancer_type",
            "patient",
            "sample",
            "biopsy_stage",
        ],
        columns="cell_type",
        values="n_cells",
        fill_value=0,
        observed=True,
    )
    .sort_index()
)

primary_percentage_wide.to_csv(
    TABLE_ROOT
    / "expanded_primary_cell_type_percentages_by_sample_wide.csv"
)
primary_count_wide.to_csv(
    TABLE_ROOT
    / "expanded_primary_cell_type_counts_by_sample_wide.csv"
)

display(
    summary
)
display(
    primary_percentage_wide
)

,pipeline_version,sample,patient,cancer_type,biopsy_stage,n_cells,n_existing_tumor,n_primary_rescued_from_unresolved,n_exploratory_rescued_from_unresolved,n_primary_tumor_epithelial_total,n_exploratory_tumor_epithelial_total,n_unresolved_before,n_unresolved_after_primary,n_unresolved_after_exploratory,raw_gene_mapping_json,high_specificity_mapping_json,scoring_info,spatial_key,annotated_h5ad,metadata_parquet
0,2026-08-03-broader-tumor-epithelial-rescue-v1,Screen_39_21,patient_39_21,NSCLC,Screen,22950,2083,1385,954,3468,4422,13658,12273,11319,"{""CAPS"": ""CAPS"", ""CDH1"": ""CDH1"", ""CEACAM5"": ""C...","{""CEACAM5"": ""CEACAM5"", ""EPCAM"": ""EPCAM"", ""ITGB...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
1,2026-08-03-broader-tumor-epithelial-rescue-v1,C2D15_39_21,patient_39_21,NSCLC,C2D15,10822,1017,207,436,1224,1660,7332,7125,6689,"{""CAPS"": ""CAPS"", ""CDH1"": ""CDH1"", ""CEACAM5"": ""C...","{""CEACAM5"": ""CEACAM5"", ""EPCAM"": ""EPCAM"", ""ITGB...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
2,2026-08-03-broader-tumor-epithelial-rescue-v1,Screen_17_26,patient_17_26,NSCLC,Screen,47896,0,7117,4807,7117,11924,40113,32996,28189,"{""CAPS"": ""CAPS"", ""CDH1"": ""CDH1"", ""CEACAM5"": ""C...","{""CEACAM5"": ""CEACAM5"", ""EPCAM"": ""EPCAM"", ""ITGB...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
3,2026-08-03-broader-tumor-epithelial-rescue-v1,C2D15_17_26,patient_17_26,NSCLC,C2D15,87913,729,12224,8864,12953,21817,75569,63345,54481,"{""CAPS"": ""CAPS"", ""CDH1"": ""CDH1"", ""CEACAM5"": ""C...","{""CEACAM5"": ""CEACAM5"", ""EPCAM"": ""EPCAM"", ""ITGB...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
4,2026-08-03-broader-tumor-epithelial-rescue-v1,Screen_18_23,patient_18_23,melanoma,Screen,72386,4193,2960,6423,7153,13576,55071,52111,45688,"{""AXL"": ""AXL"", ""BIRC7"": ""BIRC7"", ""CDH1"": ""CDH1...","{""DCT"": ""DCT"", ""GPR143"": ""GPR143"", ""MIA"": ""MIA...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
5,2026-08-03-broader-tumor-epithelial-rescue-v1,C2D15_18_23,patient_18_23,melanoma,C2D15,18594,1146,414,788,1560,2348,12169,11755,10967,"{""AXL"": ""AXL"", ""BIRC7"": ""BIRC7"", ""CDH1"": ""CDH1...","{""DCT"": ""DCT"", ""GPR143"": ""GPR143"", ""MIA"": ""MIA...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
6,2026-08-03-broader-tumor-epithelial-rescue-v1,Screen_16_22,patient_16_22,melanoma,Screen,70438,0,8692,6421,8692,15113,53259,44567,38146,"{""AXL"": ""AXL"", ""BIRC7"": ""BIRC7"", ""CDH1"": ""CDH1...","{""DCT"": ""DCT"", ""GPR143"": ""GPR143"", ""MIA"": ""MIA...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
7,2026-08-03-broader-tumor-epithelial-rescue-v1,C2D15_16_22,patient_16_22,melanoma,C2D15,76633,0,11285,7538,11285,18823,58748,47463,39925,"{""AXL"": ""AXL"", ""BIRC7"": ""BIRC7"", ""CDH1"": ""CDH1...","{""DCT"": ""DCT"", ""GPR143"": ""GPR143"", ""MIA"": ""MIA...","{'score_source': 'fresh_corrected_UCell', 'sig...",X_spatial,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
8,2026-08-03-broader-tumor-epithelial-rescue-v1,Screen_30_16,patient_30_16,melanoma,Screen,66977,0,9657,6450,9657,16107,47002,37345,30895,"{""AXL"": ""AXL"", ""BIRC7"": ""BIRC7"", ""CDH1"": ""CDH1...",

cell_type                                              B/plasma    CD4+ T  \
cancer_type  patient       sample       biopsy_stage                        
NSCLC        patient_17_26 C2D15_17_26  C2D15          0.051187  0.116024   
                           Screen_17_26 Screen         0.194171  0.202522   
             patient_39_21 C2D15_39_21  C2D15          0.000000  0.711514   
                           Screen_39_21 Screen         8.775599  1.416122   
colon_cancer patient_23_25 C2D15_23_25  C2D15         10.006834  0.253832   
                           Screen_23_25 Screen         4.958101  0.042675   
melanoma     patient_16_22 C2D15_16_22  C2D15          7.317996  0.458027   
                           Screen_16_22 Screen         8.530907  0.178881   
             patient_18_23 C2D15_18_23  C2D15          3.640959  1.027213   
                           Screen_18_23 Screen         5.299367  0.230708   
             patient_30_16 C2D15_30_16  C2D15          0.135020  0.024446   
                           Screen_30_16 Screen         2.757663  0.815205   

cell_type                                               CD8+ T  Endothelial  \
cancer_type  patient       sample       biopsy_stage                          
NSCLC        patient_17_26 C2D15_17_26  C2D15         0.116024     0.000000   
                           Screen_17_26 Screen        0.183731     0.000000   
             patient_39_21 C2D15_39_21  C2D15         0.637590     9.868786   
                           Screen_39_21 Screen        1.342048     7.002179   
colon_cancer patient_23_25 C2D15_23_25  C2D15         0.283120     8.230011   
                           Screen_23_25 Screen        0.034916     5.563315   
melanoma     patient_16_22 C2D15_16_22  C2D15         0.429319     4.932601   
                           Screen_16_22 Screen        0.171782     6.805985   
             patient_18_23 C2D15_18_23  C2D15         1.059482     8.529633   
                           Screen_18_23 Screen        0.198933     2.604095   
             patient_30_16 C2D15_30_16  C2D15         0.016487     0.131609   
                           Screen_30_16 Screen        0.879406     8.540245   

cell_type                                             Fibroblast/stromal  \
cancer_type  patient       sample       biopsy_stage                       
NSCLC        patient_17_26 C2D15_17_26  C2D15                   9.794911   
                           Screen_17_26 Screen                  9.769083   
             patient_39_21 C2D15_39_21  C2D15                   0.000000   
                           Screen_39_21 Screen                  0.000000   
colon_cancer patient_23_25 C2D15_23_25  C2D15                   0.000000   
                           Screen_23_25 Screen                  6.385785   
melanoma     patient_16_22 C2D15_16_22  C2D15                   0.000000   
                           Screen_16_22 Screen                  0.000000   
             patient_18_23 C2D15_18_23  C2D15                   4.168011   
                           Screen_18_23 Screen                  0.261100   
             patient_30_16 C2D15_30_16  C2D15                   7.996896   
                           Screen_30_16 Screen                  8.698508   

cell_type                                             Monocyte/macrophage  \
cancer_type  patient       sample       biopsy_stage                        
NSCLC        patient_17_26 C2D15_17_26  C2D15                    2.352326   
                           Screen_17_26 Screen                   4.710205   
             patient_39_21 C2D15_39_21  C2D15                    0.055443   
                           Screen_39_21 Screen                   0.000000   
colon_cancer patient_23_25 C2D15_23_25  C2D15                    9.313678   
                           Screen_23_25 Screen                   7.728119   
melanoma     patient_16_22 C2D15_16_22  C2D15                    7.722522   
                           Screen_16_22 Screen          

In [14]:
# ---------------------------------------------------------------------
# Before/after and composition plots
# ---------------------------------------------------------------------
def safe_slug(
    value: str,
) -> str:
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(
            value
        ),
    ).strip(
        "_"
    )


def save_stacked_percentage_plot(
    cancer_type: str,
) -> None:
    try:
        subset = (
            primary_percentage_wide
            .xs(
                cancer_type,
                level="cancer_type",
            )
            .copy()
        )
    except KeyError:
        return

    reset = subset.reset_index()
    reset[
        "biopsy_stage"
    ] = pd.Categorical(
        reset[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    reset = reset.sort_values(
        [
            "patient",
            "biopsy_stage",
            "sample",
        ]
    )

    cell_type_columns = [
        column
        for column in reset.columns
        if column
        not in {
            "patient",
            "sample",
            "biopsy_stage",
        }
    ]
    matrix = reset[
        cell_type_columns
    ].copy()

    ax = matrix.plot(
        kind="bar",
        stacked=True,
        figsize=(
            max(
                11,
                0.85
                * len(
                    reset
                )
                + 5,
            ),
            7,
        ),
        width=0.82,
    )
    ax.set_title(
        f"{cancer_type}: primary composition after broader tumor/epithelial rescue"
    )
    ax.set_xlabel(
        ""
    )
    ax.set_ylabel(
        "Percentage of all cells"
    )
    ax.set_ylim(
        0,
        100,
    )
    ax.set_xticklabels(
        reset[
            "sample"
        ].astype(str),
        rotation=35,
        ha="right",
    )
    ax.legend(
        title="Cell type",
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=8,
        frameon=False,
    )
    plt.tight_layout()
    plt.savefig(
        FIGURE_ROOT
        / (
            f"{safe_slug(cancer_type)}_"
            "expanded_primary_cell_type_percentages.png"
        ),
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()


def save_patient_count_plot(
    cancer_type: str,
) -> None:
    subset = primary_composition.loc[
        primary_composition[
            "cancer_type"
        ].astype(str)
        == str(
            cancer_type
        )
    ]
    if subset.empty:
        return

    pivot = (
        subset
        .pivot_table(
            index=[
                "patient",
                "biopsy_stage",
            ],
            columns="cell_type",
            values="n_cells",
            fill_value=0,
            observed=True,
        )
        .reset_index()
    )
    pivot[
        "biopsy_stage"
    ] = pd.Categorical(
        pivot[
            "biopsy_stage"
        ],
        categories=(
            BIOPSY_STAGE_ORDER
        ),
        ordered=True,
    )
    pivot = pivot.sort_values(
        [
            "patient",
            "biopsy_stage",
        ]
    )

    labels = [
        f"{patient}\n{stage}"
        for patient, stage in zip(
            pivot[
                "patient"
            ],
            pivot[
                "biopsy_stage"
            ].astype(str),
        )
    ]
    cell_type_columns = [
        column
        for column in pivot.columns
        if column
        not in {
            "patient",
            "biopsy_stage",
        }
    ]

    ax = (
        pivot[
            cell_type_columns
        ]
        .plot(
            kind="bar",
            stacked=True,
            figsize=(
                max(
                    11,
                    0.85
                    * len(
                        pivot
                    )
                    + 5,
                ),
                7,
            ),
            width=0.82,
        )
    )
    ax.set_title(
        f"{cancer_type}: paired Screen/C2D15 counts after tumor rescue"
    )
    ax.set_xlabel(
        ""
    )
    ax.set_ylabel(
        "Number of cells"
    )
    ax.set_xticklabels(
        labels,
        rotation=35,
        ha="right",
    )
    ax.legend(
        title="Cell type",
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=8,
        frameon=False,
    )
    plt.tight_layout()
    plt.savefig(
        FIGURE_ROOT
        / (
            f"{safe_slug(cancer_type)}_"
            "expanded_patient_stage_counts.png"
        ),
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()


for cancer_type in CANCER_TYPE_ORDER:
    save_stacked_percentage_plot(
        cancer_type
    )
    save_patient_count_plot(
        cancer_type
    )

# Before/after tumor and unresolved counts.
comparison = summary[
    [
        "sample",
        "cancer_type",
        "n_existing_tumor",
        "n_primary_tumor_epithelial_total",
        "n_exploratory_tumor_epithelial_total",
        "n_unresolved_before",
        "n_unresolved_after_primary",
        "n_unresolved_after_exploratory",
    ]
].copy()

comparison.to_csv(
    TABLE_ROOT
    / "tumor_unresolved_before_after_by_sample.csv",
    index=False,
)

ax = (
    comparison
    .set_index(
        "sample"
    )[
        [
            "n_existing_tumor",
            "n_primary_tumor_epithelial_total",
            "n_exploratory_tumor_epithelial_total",
        ]
    ]
    .plot(
        kind="bar",
        figsize=(15, 7),
        width=0.82,
    )
)
ax.set_title(
    "Existing versus broadened tumor/epithelial calls"
)
ax.set_xlabel(
    "Sample"
)
ax.set_ylabel(
    "Cells"
)
ax.tick_params(
    axis="x",
    rotation=35,
)
ax.legend(
    fontsize=8,
    frameon=False,
)
plt.tight_layout()
plt.savefig(
    FIGURE_ROOT
    / "existing_vs_expanded_tumor_counts.png",
    dpi=PLOT_DPI,
    bbox_inches="tight",
)
plt.close()

ax = (
    comparison
    .set_index(
        "sample"
    )[
        [
            "n_unresolved_before",
            "n_unresolved_after_primary",
            "n_unresolved_after_exploratory",
        ]
    ]
    .plot(
        kind="bar",
        figsize=(15, 7),
        width=0.82,
    )
)
ax.set_title(
    "Other/unresolved before and after tumor/epithelial rescue"
)
ax.set_xlabel(
    "Sample"
)
ax.set_ylabel(
    "Cells"
)
ax.tick_params(
    axis="x",
    rotation=35,
)
ax.legend(
    fontsize=8,
    frameon=False,
)
plt.tight_layout()
plt.savefig(
    FIGURE_ROOT
    / "unresolved_before_after_tumor_rescue.png",
    dpi=PLOT_DPI,
    bbox_inches="tight",
)
plt.close()

manifest = {
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "method": (
        "expanded cancer-specific UCell tumor/epithelial rescue "
        "restricted to current Other/unresolved cells"
    ),
    "primary_label_column": (
        "prelim_cell_type_primary_tumor_expanded"
    ),
    "exploratory_label_column": (
        "prelim_cell_type_exploratory_tumor_expanded"
    ),
    "confidence_column": (
        "tumor_epithelial_confidence_tier"
    ),
    "expanded_label": (
        EXPANDED_TUMOR_LABEL
    ),
    "malignant_vs_normal_epithelium_separated": False,
    "n_samples_completed": int(
        len(
            results
        )
    ),
    "n_samples_failed": int(
        len(
            failures
        )
    ),
    "summary": str(
        TABLE_ROOT
        / "broader_tumor_rescue_summary_by_sample.csv"
    ),
    "composition": str(
        TABLE_ROOT
        / "expanded_cell_type_counts_and_percentages_by_sample.csv"
    ),
    "before_after": str(
        TABLE_ROOT
        / "tumor_unresolved_before_after_by_sample.csv"
    ),
    "figure_root": str(
        FIGURE_ROOT
    ),
    "failures": failures,
}
write_json(
    manifest,
    OUTPUT_ROOT
    / "broader_tumor_epithelial_rescue_manifest.json",
)

display(
    comparison
)

,sample,cancer_type,n_existing_tumor,n_primary_tumor_epithelial_total,n_exploratory_tumor_epithelial_total,n_unresolved_before,n_unresolved_after_primary,n_unresolved_after_exploratory
0,Screen_39_21,NSCLC,2083,3468,4422,13658,12273,11319
1,C2D15_39_21,NSCLC,1017,1224,1660,7332,7125,6689
2,Screen_17_26,NSCLC,0,7117,11924,40113,32996,28189
3,C2D15_17_26,NSCLC,729,12953,21817,75569,63345,54481
4,Screen_18_23,melanoma,4193,7153,13576,55071,52111,45688
5,C2D15_18_23,melanoma,1146,1560,2348,12169,11755,10967
6,Screen_16_22,melanoma,0,8692,15113,53259,44567,38146
7,C2D15_16_22,melanoma,0,11285,18823,58748,47463,39925
8,Screen_30_16,melanoma,0,9657,16107,47002,37345,30895
9,C2D15_30_16,melanoma,3417,44443,75149,286392,245366,214660


# Reading and tuning the result

## Recommended final column

```python
adata.obs["prelim_cell_type_primary_tumor_expanded"]
```

The new broad class is:

```text
Tumor/epithelial
```

This includes:

```text
existing conservative tumor calls
+
primary rescued epithelial/melanoma-like cells
```

## Sensitivity column

```python
adata.obs["prelim_cell_type_exploratory_tumor_expanded"]
```

## Confidence and program records

```python
tumor_epithelial_confidence_tier
tumor_epithelial_confidence_rank

tumor_epithelial_dominant_program
tumor_epithelial_expanded_score
tumor_epithelial_expanded_percentile
tumor_epithelial_minus_max_competing_percentile

tumor_epithelial_raw_gene_count
tumor_epithelial_high_specificity_raw_support
tumor_epithelial_reference_support
```

## Making the primary tumor rescue broader

A mild adjustment is:

```python
PRIMARY_TUMOR_QUANTILE = 0.80
PRIMARY_COMPETITION_ALLOWANCE = 0.15
```

Change one parameter at a time and inspect:

```text
expanded_tumor_score_audit.png
expanded_tumor_spatial.png
tumor_unresolved_before_after_by_sample.csv
```

## Making it more specific

```python
PRIMARY_TUMOR_QUANTILE = 0.90
PRIMARY_COMPETITION_ALLOWANCE = 0.05
PRIMARY_MIN_RAW_GENES = 3
```

## Interpretation

This is a broad epithelial/tumor compartment, not a malignancy classifier.

For a future definitive tumor annotation, the strongest approach remains:

```text
H&E/pathologist/deep-learning tumor region
+
transcriptional epithelial/melanoma support
+
optional CNV evidence
```

The notebook intentionally preserves all score, support, and confidence columns
so histology-derived labels can later be combined without rerunning UCell.